In [3]:
from torch import nn
import torch
import pandas as pd
from gensim.models import KeyedVectors
import gensim.downloader as api
from torch.nn.utils.rnn import pad_sequence
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import numpy as np

In [4]:
# !pip install gensim

In [5]:
df = pd.read_csv('C:\\Users\\Lenovo\\Documents\\code\\python\\pytorch_tests\\datasets\\sentimentdataset.csv')

In [4]:
df.head()

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [5]:
df = df.set_index('Unnamed: 0')

In [6]:
X = df['Text']
Y= df['Sentiment']

In [7]:
glove = api.load("glove-wiki-gigaword-300")  # ~376MB

In [8]:
def tokenize(text):
    return text.lower().split()

In [9]:
def sentence_to_word_embeddings(sentence, model):
    tokens = tokenize(sentence)
    return [
        model[word]
        for word in tokens
        if word in model
    ]

In [10]:
X_embeddings = X.apply(lambda x: sentence_to_word_embeddings(x, glove))

In [11]:
# X_embeddings_tensors = torch.tensor( X_embeddings, dtype=torch.float32 )

X_tensors = [ torch.tensor(emb, dtype=torch.float32)   for emb in X_embeddings  ]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26908\2135117197.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  X_tensors = [ torch.tensor(emb, dtype=torch.float32)   for emb in X_embeddings  ]


In [12]:
X_padded = pad_sequence( X_tensors, batch_first=True, padding_value=0.0 )

In [13]:
print(X_padded.shape)

torch.Size([732, 22, 300])


In [14]:
lengths = torch.tensor([t.shape[0] for t in X_tensors])

In [21]:

class SentimentBiLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=512,
            num_layers=4,
            batch_first=True,
            dropout=0.3,
            bidirectional=True,
            proj_size=128
        )

        # Because bidirectional + proj_size=128
        self.classifier = nn.Linear(2 * 128, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        """
        x: (batch_size, max_seq_len, 300)
        lengths: (batch_size,)
        """

        # 1️⃣ Pack padded sequences
        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 2️⃣ LSTM forward
        packed_out, (h_n, c_n) = self.lstm(packed_x)

        # 3️⃣ Unpack
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        # out: (batch_size, max_seq_len, 256)

        # 4️⃣ Extract last valid timestep per sequence
        batch_size = out.size(0)
        last_outputs = out[
            torch.arange(batch_size),
            lengths - 1
        ]  # (batch_size, 256)

        # 5️⃣ Classification
        logits = self.classifier(self.dropout(last_outputs))

        return logits

In [22]:

class SentimentGRU(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.lstm = nn.GRU(
            input_size=300,
            hidden_size=128,
            num_layers=4,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Because bidirectional + proj_size=128
        self.classifier = nn.Linear(2 * 128, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        """
        x: (batch_size, max_seq_len, 300)
        lengths: (batch_size,)
        """

        # 1️⃣ Pack padded sequences
        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 2️⃣ LSTM forward
        packed_out, h_n = self.lstm(packed_x)

        # 3️⃣ Unpack
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        # out: (batch_size, max_seq_len, 256)

        # 4️⃣ Extract last valid timestep per sequence
        batch_size = out.size(0)
        last_outputs = out[
            torch.arange(batch_size),
            lengths - 1
        ]  # (batch_size, 256)

        # 5️⃣ Classification
        logits = self.classifier(self.dropout(last_outputs))

        return logits

In [15]:
help(nn.LSTM.__init__)

Help on function __init__ in module torch.nn.modules.rnn:

__init__(self, *args, **kwargs)
    Initialize internal Module state, shared by both nn.Module and ScriptModule.



In [16]:

num_samples = X_padded.size(0)
indices = torch.randperm(num_samples)

split = int(0.8 * num_samples)
train_idx, test_idx = indices[:split], indices[split:]

X_train = X_padded[train_idx]
X_test = X_padded[test_idx]

lengths_train = lengths[train_idx]
lengths_test = lengths[test_idx]


In [17]:
Y_enc = Y.apply(lambda x: 1 if str(x).strip().lower() =='positive' else 0)
Y_tensor = torch.tensor(Y_enc.values, dtype=torch.long)

In [18]:
y_train = Y_tensor[train_idx]
y_test = Y_tensor[test_idx]

In [19]:
X_train.shape

torch.Size([585, 22, 300])

In [20]:
y_train.shape

torch.Size([585])

In [21]:
len_tensor = torch.tensor(lengths, dtype=torch.long)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26908\1649207563.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  len_tensor = torch.tensor(lengths, dtype=torch.long)


In [22]:
len_tensor_train = len_tensor[train_idx]
len_tensor_test = len_tensor[test_idx]

In [23]:
# epoch = 50
# model = SentimentBiLSTM()
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-3,
#     weight_decay=1e-2
# )


# for e in range(epoch):
#   loss_epoch =0
#   optimizer.zero_grad()

#   logits = model(X_train[1,:,:], len_tensor_train[i])

#   loss = criterion(logits, y_train[i])
#   loss.backward()
#   optimizer.step()
#   loss_epoch += loss

#   print(f'Loss : {loss} in epoch: {e}')

In [32]:
epoch = 50
model = SentimentGRU()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

model.train()

for e in range(epoch):
    optimizer.zero_grad()

    logits = model(X_train, len_tensor_train)     # ✅ batch forward
    loss = criterion(logits, y_train)    # ✅ batch labels

    loss.backward()
    optimizer.step()

    loss_epoch = loss.item()
    print(f"Epoch {e+1}/{epoch} | Loss: {loss_epoch:.4f}")

Epoch 1/50 | Loss: 0.7069
Epoch 2/50 | Loss: 0.4985
Epoch 3/50 | Loss: 0.3374
Epoch 4/50 | Loss: 0.2356
Epoch 5/50 | Loss: 0.2204
Epoch 6/50 | Loss: 0.2413
Epoch 7/50 | Loss: 0.2556
Epoch 8/50 | Loss: 0.2523
Epoch 9/50 | Loss: 0.2437
Epoch 10/50 | Loss: 0.2241
Epoch 11/50 | Loss: 0.2036
Epoch 12/50 | Loss: 0.2011
Epoch 13/50 | Loss: 0.1988
Epoch 14/50 | Loss: 0.2049
Epoch 15/50 | Loss: 0.1995
Epoch 16/50 | Loss: 0.1874
Epoch 17/50 | Loss: 0.1731
Epoch 18/50 | Loss: 0.1562
Epoch 19/50 | Loss: 0.1439
Epoch 20/50 | Loss: 0.1267
Epoch 21/50 | Loss: 0.1131
Epoch 22/50 | Loss: 0.1080
Epoch 23/50 | Loss: 0.0982
Epoch 24/50 | Loss: 0.0873
Epoch 25/50 | Loss: 0.0850
Epoch 26/50 | Loss: 0.0775
Epoch 27/50 | Loss: 0.0728
Epoch 28/50 | Loss: 0.0644
Epoch 29/50 | Loss: 0.0616
Epoch 30/50 | Loss: 0.0581
Epoch 31/50 | Loss: 0.0473
Epoch 32/50 | Loss: 0.0440
Epoch 33/50 | Loss: 0.0379
Epoch 34/50 | Loss: 0.0319
Epoch 35/50 | Loss: 0.0250
Epoch 36/50 | Loss: 0.0272
Epoch 37/50 | Loss: 0.0235
Epoch 38/5

In [27]:
model.eval()

SentimentTransformerMHA(
  (input_proj): Identity()
  (pos_embedding): Embedding(22, 300)
  (layers): ModuleList(
    (0-1): 2 x MHAEncoderLayer(
      (mha): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=300, out_features=300, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=300, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=300, bias=True)
      )
      (norm1): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=300, out_features=150, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=150, out_features=2, bias=True)
  )
)

In [28]:
logits = model(X_test, len_tensor_test)

In [29]:
logits.shape

AttributeError: 'tuple' object has no attribute 'shape'

In [30]:
model.eval()
eval_loss = 0.0

with torch.no_grad():
  logits = model(X_test, len_tensor_test)
  loss = criterion(logits, y_test)

  print(f'Loss: {loss.item()}')

TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not tuple

In [ ]:
model.eval()

pred = []

with torch.no_grad():
    logits = model(X_test, len_tensor_test)
    pred = torch.argmax(logits, dim=1)

print(pred)

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])


In [33]:
results= pd.DataFrame(np.array(pred), columns=['pred'])

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26908\401190076.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  results= pd.DataFrame(np.array(pred), columns=['pred'])


In [34]:
results['test'] = pd.Series(np.array(y_test))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26908\1029559609.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  results['test'] = pd.Series(np.array(y_test))


In [ ]:
from sklearn.metrics import confusion_matrix

# Ensure on CPU + numpy
y_true = y_test.detach().cpu().numpy()
y_pred = pred.detach().cpu().numpy()

cm = confusion_matrix(y_true, y_pred)
cm

array([[134,   2],
       [  5,   6]])

In [42]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class MHAPooling(nn.Module):
    """
    Multi-head attention pooling with a learnable query vector
    """
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        # Learnable global query (1 per head implicitly)
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x, lengths):
        """
        x: (B, T, D)
        lengths: (B,)
        """

        B, T, D = x.size()

        # Expand query for batch
        query = self.query.expand(B, -1, -1)  # (B, 1, D)

        # Padding mask: True = ignore
        key_padding_mask = (
            torch.arange(T, device=lengths.device)
            .unsqueeze(0) >= lengths.unsqueeze(1)
        )  # (B, T)

        # Multi-head attention
        pooled, attn_weights = self.mha(
            query=query,           # (B, 1, D)
            key=x,                 # (B, T, D)
            value=x,               # (B, T, D)
            key_padding_mask=key_padding_mask
        )

        # pooled: (B, 1, D)
        pooled = pooled.squeeze(1)  # (B, D)

        return pooled, attn_weights


In [24]:

class MHAEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask):
        # Self‑Attention
        attn_out, attn_weights = self.mha(
            x, x, x,
            key_padding_mask=padding_mask,
            need_weights=True,
            average_attn_weights=False
        )

        x = self.norm1(x + self.dropout(attn_out))

        # Feed Forward
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x, attn_weights


In [25]:
class SentimentTransformerMHA(nn.Module):
    def __init__(
        self,
        vocab_size=None,
        embed_dim=128,
        num_heads=4,
        ff_dim=256,
        num_layers=2,
        num_classes=2,
        max_len=512,
        padding_idx=0,
        dropout=0.2,
        input_dim=None,
        pretrained_embeddings=None
    ):
        super().__init__()

        self.use_embedding_layer = vocab_size is not None
        self.embed_dim = embed_dim

        if self.use_embedding_layer:
            # 1️⃣ Token Embeddings
            self.embedding = nn.Embedding(
                vocab_size, embed_dim, padding_idx=padding_idx
            )
            if pretrained_embeddings is not None:
                self.embedding.weight.data.copy_(pretrained_embeddings)
            self.input_proj = None
        else:
            self.embedding = None
            if input_dim is None or input_dim == embed_dim:
                self.input_proj = nn.Identity()
            else:
                self.input_proj = nn.Linear(input_dim, embed_dim)

        # Positional Embeddings
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        # 2️⃣ Encoder Stack
        self.layers = nn.ModuleList([
            MHAEncoderLayer(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # 3️⃣ Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, inputs, lengths):
        """
        inputs: [B, T] token ids or [B, T, D] embeddings
        lengths: [B]
        """

        if self.use_embedding_layer:
            B, T = inputs.shape
            token_emb = self.embedding(inputs)
        else:
            B, T, D = inputs.shape
            token_emb = self.input_proj(inputs)

        # Padding mask (True = ignore position)
        padding_mask = torch.arange(T, device=lengths.device)\
            .expand(B, T) >= lengths.unsqueeze(1)

        positions = torch.arange(T, device=inputs.device)\
            .unsqueeze(0).expand(B, T)
        pos_emb = self.pos_embedding(positions)

        x = token_emb + pos_emb

        all_attn_weights = []

        # 2️⃣ Encoder Layers
        for layer in self.layers:
            x, attn_weights = layer(x, padding_mask)
            all_attn_weights.append(attn_weights)

        # Stack attention weights
        all_attn_weights = torch.stack(all_attn_weights)

        # 3️⃣ Attention‑aware Pooling (mask‑aware mean)
        mask = (~padding_mask).unsqueeze(-1)
        x = x * mask
        pooled = x.sum(dim=1) / mask.sum(dim=1)

        # 4️⃣ Classifier
        logits = self.classifier(pooled)

        return logits, all_attn_weights

In [26]:
epoch = 50
model = SentimentTransformerMHA(
    vocab_size=None,
    embed_dim=300,
    input_dim=300,
    max_len=X_train.size(1),
    num_heads=4,
    ff_dim=512,
    num_layers=2,
    dropout=0.2
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

model.train()

for e in range(epoch):
    optimizer.zero_grad()

    logits, _ = model(X_train, len_tensor_train)     # ✅ batch forward
    loss = criterion(logits, y_train)    # ✅ batch labels

    loss.backward()
    optimizer.step()

    loss_epoch = loss.item()
    print(f"Epoch {e+1}/{epoch} | Loss: {loss_epoch:.4f}")

Epoch 1/50 | Loss: 0.7697
Epoch 2/50 | Loss: 0.3368
Epoch 3/50 | Loss: 0.2338
Epoch 4/50 | Loss: 0.2399
Epoch 5/50 | Loss: 0.2337
Epoch 6/50 | Loss: 0.2161
Epoch 7/50 | Loss: 0.1941
Epoch 8/50 | Loss: 0.2076
Epoch 9/50 | Loss: 0.1805
Epoch 10/50 | Loss: 0.1621
Epoch 11/50 | Loss: 0.1389
Epoch 12/50 | Loss: 0.1441
Epoch 13/50 | Loss: 0.1377
Epoch 14/50 | Loss: 0.1323
Epoch 15/50 | Loss: 0.1262
Epoch 16/50 | Loss: 0.1256
Epoch 17/50 | Loss: 0.1173
Epoch 18/50 | Loss: 0.1153
Epoch 19/50 | Loss: 0.1090
Epoch 20/50 | Loss: 0.1158
Epoch 21/50 | Loss: 0.0851
Epoch 22/50 | Loss: 0.1078
Epoch 23/50 | Loss: 0.0971
Epoch 24/50 | Loss: 0.0804
Epoch 25/50 | Loss: 0.0828
Epoch 26/50 | Loss: 0.0636
Epoch 27/50 | Loss: 0.0600
Epoch 28/50 | Loss: 0.0584
Epoch 29/50 | Loss: 0.0474
Epoch 30/50 | Loss: 0.0559
Epoch 31/50 | Loss: 0.0444
Epoch 32/50 | Loss: 0.0462
Epoch 33/50 | Loss: 0.0379
Epoch 34/50 | Loss: 0.0411
Epoch 35/50 | Loss: 0.0397
Epoch 36/50 | Loss: 0.0355
Epoch 37/50 | Loss: 0.0334
Epoch 38/5

In [36]:
model.eval()
eval_loss = 0.0

with torch.no_grad():
  logits, _ = model(X_test, len_tensor_test)
  loss = criterion(logits, y_test)

  print(f'Loss: {loss.item()}')

Loss: 0.27997592091560364


In [37]:
from sklearn.metrics import confusion_matrix

# Ensure on CPU + numpy
y_true = y_test.detach().cpu().numpy()
y_pred = pred.detach().cpu().numpy()

cm = confusion_matrix(y_true, y_pred)
cm

array([[134,   2],
       [  5,   6]])

## State-of-the-art Sentiment Model (PyTorch + DeBERTa-v3)
This section fine-tunes `microsoft/deberta-v3-base` for binary sentiment classification using your `Text` and `Sentiment` columns.

Run the cells below in order.

In [38]:
# If needed, install dependencies
# %pip install -U transformers datasets scikit-learn tqdm

  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp313-cp313-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.23.0-cp313-cp313-win_amd64.whl.metadata (82 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ------ ----

In [1]:
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

c:\Users\Lenovo\Documents\code\python\pytorch_tests\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [6]:
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

work_df = df[["Text", "Sentiment"]].copy()
work_df["Text"] = work_df["Text"].astype(str).fillna("")
work_df["label"] = work_df["Sentiment"].astype(str).str.strip().str.lower().map(
    {"negative": 0, "positive": 1}
)
work_df = work_df.dropna(subset=["label"]).reset_index(drop=True)
work_df["label"] = work_df["label"].astype(int)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    work_df["Text"].tolist(),
    work_df["label"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=work_df["label"].tolist(),
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SentimentTextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = SentimentTextDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = EPOCHS * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Train size: {len(train_ds)} | Val size: {len(val_ds)}")

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 35509.99it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.

Train size: 39 | Val size: 10


In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Validation", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        logits = outputs.logits
        loss = outputs.loss

        preds = torch.argmax(logits, dim=-1)

        running_loss += loss.item()
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = running_loss / max(len(dataloader), 1)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="binary")

    return avg_loss, acc, f1, all_labels, all_preds


best_f1 = -1
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    model.to(device)

print(f"Best validation F1: {best_f1:.4f}")

Training:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader, device)
print(f"Final validation loss: {val_loss:.4f}")
print(f"Final validation accuracy: {val_acc:.4f}")
print(f"Final validation F1: {val_f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))


def predict_sentiment(texts, model, tokenizer, device, max_len=256):
    model.eval()
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        truncation=True,
        max_length=max_len,
        padding=True,
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

    label_map = {0: "negative", 1: "positive"}
    return [
        {
            "text": text,
            "pred_label": label_map[int(pred.item())],
            "confidence": float(prob.max().item()),
            "prob_negative": float(prob[0].item()),
            "prob_positive": float(prob[1].item()),
        }
        for text, pred, prob in zip(texts, preds, probs)
    ]


sample_results = predict_sentiment(
    [
        "This movie was excellent and emotionally engaging.",
        "The product quality is terrible and I want a refund.",
    ],
    model,
    tokenizer,
    device,
    max_len=MAX_LEN,
)

pd.DataFrame(sample_results)